In [1]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()

In [2]:
documents = []

for file in files:
    doc = file.parse()
    documents.append(doc)

In [3]:
len(documents)

72

In [4]:
from minsearch import Index

index = Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)

index.fit(documents)

In [5]:
question = "How does the agentic loop keep calling the model until it stops?"

search_results = index.search(
    question,
    num_results=5
)

search_results

[{'content': '# The Agentic Loop\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=ePlQUcTPPjw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn the previous lesson, we did function calling by hand. We sent a\nmessage and got back a function call. We ran it, sent the result back,\nand got the answer.\n\nThat works for one function call. It breaks down when the model wants\nto search several times, or when the first search misses the answer.\nWe don\'t know in advance how many calls the model will want. So we\nneed a loop that keeps calling the model and running tools until it\'s\ndone. An agent is exactly that.\n\n## Anatomy of an agent\n\nWith the LLM in the driver\'s seat, we have an agent. It\'s an AI\nassistant whose goal is to help the user.\n\nAn agent has three parts:\n\n- Instructions, the role and behavior we want. We pass this as the\n  `developer` message. The better the instructions, the better the\n  agent helps.\n- Tools, the functions the agent can call to carry

In [6]:
import os 
from rag_helper import RAGBase
from google import genai

client = genai.Client(api_key=os.getenv("GEMINI_API_KEY"))

assistant = RAGBase(
    index=index,
    llm_client=client,
)

answer, tokens = assistant.rag("How does the agentic loop keep calling the model until it stops?")
print(answer)
print(tokens)

Tokens - Input: 7933, Output: 349
The agentic loop keeps calling the model until it stops by using a `while` loop that continuously interacts with the model based on its responses.

Here's how it works:

1.  **Initial Call:** The loop starts by sending the initial instructions and user question to the model.
2.  **Processing Response:** The model processes the input and returns a response.
3.  **Check for Function Calls:** The agent then checks this response.
    *   **If the response contains a function call:**
        *   The agent executes the requested function (e.g., `search`).
        *   The output of the function call is appended to the message history.
        *   The `has_function_calls` flag is set to `True`.
        *   The loop continues, sending the updated message history (including the tool's output) back to the model in the next iteration. This allows the model to see the results of its requested action.
    *   **If the response does NOT contain any function calls:**


In [7]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)

In [8]:
len(chunks)

295

In [9]:
index = Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)

index.fit(chunks)

In [10]:
client = genai.Client(api_key=os.getenv("GEMINI_API_KEY"))

assistant = RAGBase(
    index=index,
    llm_client=client,
)

answer, tokens = assistant.rag("How does the agentic loop keep calling the model until it stops?")
print(answer)
print(tokens)

Tokens - Input: 2585, Output: 348
The agentic loop keeps calling the model using a `while True` loop. Inside this loop, the model is called with the current message history.

Here's how it works and stops:

1.  **Continuous Calling:** The `while True` loop ensures that the model (`openai_client.responses.create`) is continuously called in each iteration.
2.  **Checking for Function Calls:** After receiving a response from the model, the code iterates through `response.output` to check if any items are of `type == "function_call"`.
3.  **`has_function_calls` Flag:** A boolean flag, `has_function_calls`, is set to `True` if any function calls are detected in the model's response. If no function calls are found, this flag remains `False`.
4.  **Executing Tools (if any):** If function calls are detected, `make_call()` is executed, and its output is appended to the message history, which will be sent to the model in the next iteration.
5.  **Stopping Condition:** At the end of each loop ite

In [12]:
instructions = '''
You're a course teaching assistant. Answer the student's question using the search tool. Make multiple searches with different keywords before answering.'''

In [11]:
from langchain_core.tools import tool

@tool
def search(query: str) -> dict[str, str]:
    """
    Search the FAQ database for entries matching the given query.
    """
    return index.search(
        query,
        num_results=5,
    )

In [13]:
from langchain.chat_models import init_chat_model

model = init_chat_model(
    "gemini-2.5-flash",
    model_provider="google-genai",
    temperature=0.5,
    timeout=600,
    max_tokens=25000,
    streaming=True,
)

In [16]:
from langchain.agents import create_agent

agent = create_agent(
    model=model,
    tools=[search],
    system_prompt=instructions,
)

result = agent.invoke(
    {"messages": [{"role": "user", "content": "How does the agentic loop work, and how is it different from plain RAG?"}]}
)
print(result["messages"][-1].text)

history = result.get("intermediate_steps", [])

# Since 'intermediate_steps' tracks every action taken:
total_calls = len(history)
print(total_calls)

The agentic loop and plain Retrieval Augmented Generation (RAG) are both techniques that enhance Large Language Models (LLMs), but they differ significantly in their operational flow, decision-making capabilities, and interaction with tools.

### How the Agentic Loop Works

The agentic loop is a dynamic and iterative process where an LLM acts as an "agent" that can reason, plan, execute actions, and refine its approach based on intermediate results. It typically follows these steps in a continuous loop:

1.  **LLM Call:** The LLM receives a prompt or a goal.
2.  **Decision Making & Tool Use:** Based on the prompt, the LLM decides whether it needs to use external tools (like a search engine, code interpreter, or a database query tool) to gather more information or perform specific actions.
3.  **Tool Execution:** If a tool is chosen, the agent executes it.
4.  **Result Integration:** The output or results from the tool execution are fed back to the LLM.
5.  **Iteration or Final Answer:*